# Tutorial: Plan 004.3 Materialization and Reductions

**Audience:** Python users who have completed the current-file query introduction.

**Prerequisites:** an Archiver development environment, basic Path use, and familiarity with Catalog.files().

**Learning goals:** use first(), complete reductions, bounded streaming, and deliberately unbounded streaming without reaching into Archiver's private SQL implementation.

**Outline:** setup → bounded preview → first and reductions → streams → exercise → pitfalls and cleanup.

Every cell uses a disposable source directory and catalog. Run the final cleanup cell when you are done.

## 1. Create a small current catalog

The setup creates three text files and one binary file. Their sizes make the effects of sorting, limits, and reductions visible.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog
from archiver.query import mtime, path, size


temporary_directory = TemporaryDirectory(prefix="archiver-materialization-")
workspace = Path(temporary_directory.name)
root = workspace / "source"
root.mkdir()


def write_example(relative_path: str, contents: bytes, mtime_ns: int) -> None:
    file_path = root / relative_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_bytes(contents)
    os.utime(file_path, ns=(mtime_ns, mtime_ns))


write_example("a-readme.txt", b"a", 1_700_000_000_000_000_000)
write_example("b-summary.txt", b"bb", 1_700_000_001_000_000_000)
write_example("nested/c-report.txt", b"ccc", 1_700_000_002_000_000_000)
write_example("nested/data.bin", b"dddd", 1_700_000_003_000_000_000)

catalog = Catalog.create(workspace / "catalog.sqlite")
catalog.scan_directory(root)
print("Cataloged", len(catalog.current_files(root)), "current observations")


## 2. Build one bounded preview

The query is immutable and lazy. It selects text files, orders them largest-first, and permits at most two rows when a bounded row terminal runs.

In [ ]:
preview = catalog.files(root).where(path.glob("*.txt")).sort(size.desc()).limit(2)

preview_paths = [row.relative_path.as_posix() for row in preview.fetch()]
assert preview_paths == ["nested/c-report.txt", "b-summary.txt"]
preview_paths


## 3. Ask for one row or complete reductions

first() uses the same filtering and ordering, but asks SQLite for one row. count() and total_size_bytes() deliberately ignore the display limit: they describe every matching current observation, not only the two-row preview.

In [ ]:
largest_text = preview.first()
assert largest_text is not None
assert largest_text.relative_path.as_posix() == "nested/c-report.txt"

summary = {
    "first_path": largest_text.relative_path.as_posix(),
    "complete_match_count": preview.count(),
    "complete_total_bytes": preview.total_size_bytes(),
    "preview_row_count": len(preview.fetch()),
}
assert summary == {
    "first_path": "nested/c-report.txt",
    "complete_match_count": 3,
    "complete_total_bytes": 6,
    "preview_row_count": 2,
}
summary


## 4. Stream the bounded preview

stream() yields typed observations one at a time, in the same deterministic order as fetch(), and retains the query's row limit. Keep the catalog open until the iterator has been consumed.

In [ ]:
bounded_paths = [row.relative_path.as_posix() for row in preview.stream()]
assert bounded_paths == preview_paths
bounded_paths


## 5. Make unbounded consumption explicit

stream_all() removes only the row bound; it retains this query's filters and ordering. It is intentionally named to make full traversal visible in code. There is no fetch_all().

In [ ]:
all_text_paths = [row.relative_path.as_posix() for row in preview.stream_all()]
assert all_text_paths == [
    "nested/c-report.txt",
    "b-summary.txt",
    "a-readme.txt",
]
all_text_paths


## Exercise

Build a query that finds files under nested/, orders them by oldest observed modification time, and returns only its first result. Predict the path before running the solution.

In [ ]:
nested_oldest = catalog.files(root).where(path.glob("nested/*")).sort(mtime.asc()) # .limit(1): not needing this actually wince we use first
nested_first = nested_oldest.first()

assert nested_first is not None
assert nested_first.relative_path.as_posix() == "nested/c-report.txt"
{
    "first_nested_path": nested_first.relative_path.as_posix(),
    "nested_count": nested_oldest.count(),
    "nested_total_bytes": nested_oldest.total_size_bytes(),
}


## Pitfall and extension

**Pitfall:** do not infer totals from len(query.fetch()); fetch() is bounded. Use count() or total_size_bytes() for complete filtered totals.

**Extension:** add another where(path.glob(...)) predicate. Repeated predicates are combined with logical AND, and all terminals will use the refined query.

In [ ]:
catalog.close()
temporary_directory.cleanup()
print("Cleaned up the disposable catalog and source directory.")
